In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import numpy as np
import torch
import pprint

# add parent dir so importing top level files works in notebook subdir
parent_dir = str(Path().resolve().parent)
sys.path.insert(0, parent_dir)

from llm import qwen_utils, tools
from autoencoder.model_qwen import QwenAutoencoder

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [10]:
graph_dir = Path("output/aenoisy_states15_iter10k/video01_00240/graph")
preprocessed_dir = Path("data/preprocessed/qwen3_da3_subsampled/video01_00240")
ae_path = Path("data/preprocessed/qwen3_da3_subsampled/video01_00240/autoencoder_noise/best_ckpt.pth")

print(f"Graph dir: {graph_dir}")
print(f"Autoencoder path: {ae_path}")

Graph dir: output/aenoisy_states15_iter10k/video01_00240/graph
Autoencoder path: data/preprocessed/qwen3_da3_subsampled/video01_00240/autoencoder_noise/best_ckpt.pth


In [11]:
# Load autoencoder
autoencoder = QwenAutoencoder(input_dim=20480, latent_dim=16).to("cuda")
autoencoder.load_state_dict(torch.load(ae_path, map_location="cuda"))
autoencoder.eval()
print(f"Autoencoder loaded from {ae_path}")

# Load graph data and create toolkit (accepts npz directly)
toolkit = tools.GraphTools(
    positions=np.load(graph_dir / "positions.npy"),
    clusters=np.load(graph_dir / "clusters.npy"),
    centroids=np.load(graph_dir / "c_centroids.npy"),
    centers=np.load(graph_dir / "c_centers.npy"),
    extents=np.load(graph_dir / "c_extents.npy"),
    adjacency=np.load(graph_dir / "graph.npy"),
    bhattacharyya_coeffs=np.load(graph_dir / "bhattacharyya_coeffs.npy"),
    qwen_feats=np.load(graph_dir / "c_qwen_feats.npz"),
    patch_latents_through_time=np.load(graph_dir / "patch_latents_through_time.npy"),
    autoencoder=autoencoder,
)
graph_tools = toolkit.get_all_tools()
print(f"Loaded {len(graph_tools)} tools: {list(graph_tools.keys())}")
print(f"Graph has {len(np.unique(toolkit.clusters))} nodes, {toolkit.adjacency.shape[0]} timesteps")

Autoencoder loaded from data/preprocessed/qwen3_da3_subsampled/video01_00240/autoencoder_noise/best_ckpt.pth
Loaded 9 tools: ['node_distances_through_time', 'node_overlap_scores_through_time', 'node_overlap_position_at_time', 'inspect_highres_node_at_time', 'inspect_node_through_time', 'inspect_scene_at_time', 'voxelize_scene', 'node_movement', 'relative_movement']
Graph has 8 nodes, 20 timesteps


In [4]:
if "model" in locals():
    del model # pyright: ignore # noqa
if "processor" in locals():
    del processor # pyright: ignore # noqa
# Load patched Qwen3-VL model with custom feature injection support
model, processor = qwen_utils.get_patched_qwen3(size="32B", use_fp8=False)
print(f"Model loaded on device: {model.device}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

model-00007-of-00014.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00002-of-00014.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

Model loaded on device: cuda:0


# Raw Features

In [14]:
# first scene
grasper = toolkit.qwen_feats["0"][0]
liver = toolkit.qwen_feats["1"][0]
gallbladder = toolkit.qwen_feats["2"][0]

messages = [
    {"role": "system", "content": [{"type": "text", "text": "Your are a helpful assistant."}]},
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Image:"},
            {"type": "image", "image": None},
            {"type": "text", "text": "\n\nWhat do you see in the image?"},
        ],
    },
]

grasper_feats = torch.tensor(grasper, device=model.device).float()
liver_feats = torch.tensor(liver, device=model.device).float()
gallbladder_feats = torch.tensor(gallbladder, device=model.device).float()

qwen_utils.generate_with_vision_features(
    messages=messages,
    vision_features=[
        gallbladder_feats
    ],
    model=model,
    processor=processor,
)


"So, let's look at the image. First, it's a close-up of what seems to be a medical or surgical scene. There's a structure that looks like a tube or vessel, maybe a blood vessel or part of an organ. The tissue is reddish, which is typical of internal organs. There's a metallic instrument, probably a surgical tool like a forceps or clamp, holding or manipulating the tissue. The background has more tissue with visible blood vessels and some white spots, maybe fat or connective tissue. The main elements are the tissue, the surgical instrument, and the surrounding anatomical structures. Let me describe it step by step: the central part is a cylindrical structure (like a vein or artery) with a smooth surface, some small blood vessels on it. The instrument is metallic, holding the tissue, and the background is red, fleshy tissue with visible capillaries and maybe some white deposits. So, the image shows a surgical or endoscopic view of internal tissue with a surgical instrument in use.\n</thi

# Node classification

In [5]:
# classify_tool_names = [
#     "node_distances_through_time",
#     "node_overlap_scores_through_time",
#     "node_movement",
# ]
classify_tool_names = ["voxelize_scene"]

classify_tool_call_limits = {
    "voxelize_scene": 1,
}

classify_tool_dict = toolkit.get_tools_by_name(classify_tool_names)

classify_system_prompt = """
# Task
- You are an expert visceral surgeon analyzing a cholecystectomy scene.
- You have access to tools that represent the scene as a dynamic 3D scene graph or voxelized.
- Your initial scene representation is a set of graph nodes at a single timestep.

# Scene representation
- The scene represents a brief (2-3 seconds) excerpt from a laparoscopic cholecystectomy.
- The scene consists of 20 timesteps.
- The scene is reconstructed from a laparoscopic camera view.
- Coordinate system:
    - relative to the laparoscopic camera
    - laparoscopic camera looks in positive z direction
    - axes are normalized, distances are relative and do not correspond to real world units
    - increase in x means movement to the right
    - increase in y means movement to the bottom
    - increase in z means movement away from the camera
- The nodes were created by a clustering algorithm and are not perfect.
    - Single scene objects might be split into multiple nodes.
    - The nodes' rough images can contain global scene context, but should be focused on the respective object the node represents.

# Rules
- Strictly respect tool call limits. (Initially given by user prompt, updated by tool responses.)
- Think briefly, optimize for answer speed.
"""

classify_prompt = """
Find out which and how many objects are actually present in the scene and assign each node to a scene object. 
"""

# Use your tools to consider the following information of each node:
# - the node's rough image
# - the node's position relative to other nodes
# - the node's size relative to other nodes
# - the node's movement, on its own and relative to other nodes
# """

timestep_idx = 0

In [6]:
Path("output/classify.rrd").unlink(missing_ok=True)
toolkit.start_recording("output/classify.rrd")
response = qwen_utils.prompt_graph_agent(
    question=classify_prompt,
    node_feats=toolkit.qwen_feats,
    initial_timestep_idx=timestep_idx,
    node_centers=toolkit.point_o2n(toolkit.centers),
    node_centroids=toolkit.point_o2n(toolkit.centroids),
    node_extents=toolkit.distance_o2n(toolkit.extents),
    model=model,
    processor=processor,
    system_prompt=classify_system_prompt,
    tools=classify_tool_dict,
    tool_call_limits=classify_tool_call_limits,
    verbose=True,
    max_new_tokens=10000,
    max_thinking_tokens=8000,
    zero_positional_encodings=True,
)
toolkit.stop_recording()


[11:48:10] --- Iteration 0 ---
[11:50:20] [Assistant Response]:
Okay, let's try to figure out how many objects are present in this cholecystectomy scene and assign each node to a specific object. First, I need to recall that the scene is from a laparoscopic view, so it's a 3D space with the camera looking along the positive z-axis. The nodes are clustered, but the clustering might not be perfect, so some objects could be split into multiple nodes.

Looking at the nodes provided for timestep 0, there are 13 nodes. Each node has a rough image, centroid, and a bounding box. The rough images show different parts of the surgical scene. Let me go through each node one by one.

Node 0: The image shows a part of a surgical instrument, maybe a clamp or a hook, with some tissue. The centroid is at (12.29, -1.61, 36.89). The bounding box is relatively small (6.35 x 17.32 x 8.75), which suggests it's a specific part of an instrument or a small tissue section.

Node 1: The image looks like a tube 

In [10]:
print("tok/s", response["tok_per_sec"], "total_time", response["total_time"], "total_generation_time", response["total_generation_time"])

tok/s 20.709274793915906 total_time 357.13731956481934 total_generation_time 356.0723431110382


# Spatial

In [ ]:
spatial_tool_names = [
    "node_overlap_position_at_time",
    "inspect_highres_node_at_time",
    "voxelize_scene",
]

spatial_tool_call_limits = {
    "inspect_highres_node_at_time": 5,
    "voxelize_scene": 5,
}

spatial_tool_dict = toolkit.get_tools_by_name(spatial_tool_names)

spatial_system_prompt = """
# Task
You are an expert visceral surgeon locating objects or actions in a cholecystectomy scene.
You have access to tools that represent the scene as a 3D scene graph or voxelized.
Your initial scene representation is a set of graph nodes at the relevant timestep.

# Scene representation
- Coordinate system:
    - increase in x means movement to the right
    - increase in y means movement to the bottom
    - increase in z means movement away from the camera
- The nodes were created by a clustering algorithm and might be imperfect (single object split into multiple nodes / multiple objects merged into single node).

# Rules
- Strictly respect tool call limits. (Initially given by user prompt, updated by tool responses.)
- Before starting with the task, think about which nodes represent which objects.

# Answer format
Your final answer should always contain exactly one 3D point as a JSON object. Never refuse!
Report the point in the graph's 3D coordinate system.
"""

spatial_prompt = """
Identify the 3D location of 'grasper grasping gallbladder' in the scene.

Example output (format only): ... {"x": X, "y": Y, "z": Z} ...
"""

timestep_idx = 0

In [ ]:
Path("output/spatial.rrd").unlink(missing_ok=True)
toolkit.start_recording("output/spatial.rrd")
response = qwen_utils.prompt_graph_agent(
    question=spatial_prompt,
    node_feats=toolkit.qwen_feats,
    initial_timestep_idx=timestep_idx,
    node_centers=toolkit.point_o2n(toolkit.centers),
    node_centroids=toolkit.point_o2n(toolkit.centroids),
    node_extents=toolkit.distance_o2n(toolkit.extents),
    model=model,
    processor=processor,
    system_prompt=spatial_system_prompt,
    tools=spatial_tool_dict,
    tool_call_limits=spatial_tool_call_limits,
    verbose=True,
)


[17:54:20] --- Iteration 0 ---
[17:57:29] [Assistant Response]:
Okay, let me try to figure out what's going on here. The user provided a scene representation from a cholecystectomy, which is a surgery to remove the gallbladder. They have a list of nodes with their centroids, bounding boxes, and rough images. My job is to identify which objects these nodes correspond to in the context of the surgery.

First, I need to recall typical components in a laparoscopic cholecystectomy. Common elements include the gallbladder, surgical instruments like graspers, dissectors, the camera, maybe the liver, cystic duct and artery, and perhaps some fat or connective tissue. The laparoscopic tools are usually metallic instruments with handles, and the visual would show the internal organs.

Looking at the nodes:

Node 0: centroid at (1.22, -1.51, 0.6). The rough image shows a metallic tool. Since it's a laparoscopic scene, this is likely a surgical instrument, maybe a grasper or clamp. The position is

In [31]:
print("tok/s", response["tok_per_sec"], "total_time", response["total_time"], "total_generation_time", response["total_generation_time"])

tok/s 23.29436320844027 total_time 103.89036440849304 total_generation_time 102.77164387702942


In [44]:
toolkit.log_final_prediction(
    position=toolkit.point_n2o(np.array([0.0452, 0.0635, -0.1261])),
    timestep_idx=timestep_idx,
    label="grasper grasping gallbladder",
    entity_name="zz_final_prediction",
)
toolkit.stop_recording()

# Temporal

In [ ]:
temporal_tool_names = [
    "node_distances_through_time",
    "node_overlap_scores_through_time",
    "inspect_node_through_time",
    "inspect_scene_at_time",
    "inspect_highres_node_at_time",
    "voxelize_scene",
]

temporal_tool_call_limits = {
    "inspect_node_through_time": 5,
    "inspect_scene_at_time": 3,
    "inspect_highres_node_at_time": 5,
    "voxelize_scene": 5,
}

temporal_tool_dict = toolkit.get_tools_by_name(temporal_tool_names)

# tool_dict = toolkit.get_tools_by_name(["voxelize_scene"])
# tool_call_limits = {"voxelize_scene": 5}

temporal_system_prompt = """
# Task
You are an expert visceral surgeon analyzing a cholecystectomy scene for temporal reasoning.
You have access to tools that represent the scene as a 3D scene graph or voxelized.
Your initial scene representation is a set of graph nodes at timestep 0.

# Scene representation
- Coordinate system:
    - increase in x means movement to the right
    - increase in y means movement to the bottom
    - increase in z means movement away from the camera

# Rules
- Strictly respect tool call limits. (Initially given by user prompt, updated by tool responses.)

# Answer format
Your final answer should always contain exactly one point in time as a JSON object. Never refuse!
Report the point in time as a timestep (integer).
"""

temporal_prompt = """
Identify the point in time when '{question}' occurs.

The graph has 20 timesteps (t=0 to t=19}).

Example output (format only): ... {{"timestep": T}} ...
"""

In [ ]:
system_prompt = """
# Task
You are an expert visceral surgeon analyzing a cholecystectomy scene for temporal reasoning.
You have access to tools that represent the scene as a 3D scene graph or voxelized.
Your initial scene representation is a set of graph nodes at timestep 0.

# Scene representation
- Coordinate system:
    - increase in x means movement to the right
    - increase in y means movement to the bottom
    - increase in z means movement away from the camera

# Rules
- Strictly respect tool call limits. (Initially given by user prompt, updated by tool responses.)
- Reason very briefly in each turn and optimize for answer speed.

# Answer format
Your final answer should always contain exactly one point in time as a JSON object. Never refuse!
Report the point in time in seconds.
"""

prompt = """
Identify a point in time to answer the question: 'When does the grasper start grasping the gallbladder?'.

The graph has 20 timesteps (t=0 to t=19).

Example output (format only): ... {"timestep": X} ...
"""

# system_prompt = "You are an expert visceral surgeon analyzing a cholecystectomy scene for temporal reasoning. We are currently testing your capabilities to interact with the scene graph. Reason briefly, optimize for answer speed."

# prompt = "Do NOT call any tools, just tell be a joke. I want to test if inference speed is slow or if the tool too is slow."

In [ ]:
# Clear CUDA cache before generation to avoid OOM
torch.cuda.empty_cache()

toolkit.start_recording("temporal.rrd")
response = qwen_utils.prompt_graph_agent(
    question=prompt,
    node_feats=toolkit.qwen_feats,
    initial_timestep_idx=0,
    node_centers=toolkit.point_o2n(toolkit.centers),
    node_centroids=toolkit.point_o2n(toolkit.centroids),
    node_extents=toolkit.distance_o2n(toolkit.extents),
    model=model,
    processor=processor,
    system_prompt=system_prompt,
    tools=tool_dict,
    tool_call_limits=tool_call_limits,
    verbose=True,
)
toolkit.stop_recording()
# if type(response) == dict:
#     pprint.pprint(response["message_history"])
# else:
#     print(response)


[13:10:13] --- Iteration 0 ---
[13:11:34] [Assistant Response]:
Okay, I need to figure out when the grasper starts grasping the gallbladder in this cholecystectomy scene. Let's see. The user provided a graph with nodes at timestep 0, and there are 20 timesteps total from 0 to 19.

First, I should identify which nodes represent the grasper and the gallbladder. The nodes have lowres-visual-descriptor images. The grasper is probably a surgical instrument, so looking at the descriptors: nodes 1, 3, 7, 8, 10 might be instruments. Wait, node 1 has a "clamp" or "grasper" look in its descriptor image. Similarly, node 3 has a similar instrument but maybe different. Wait, node 1's descriptor is "a pair of clamps" or something, maybe a grasper. The gallbladder is likely a structure with a textured appearance, maybe the nodes with more organic structures, like node 2, 4, 5, 8, etc.

Wait, node 2's descriptor is "a fleshy structure with visible blood vessels," which could be the gallbladder. Node 

# Check tool call recording

In [ ]:
# Test tool call logging with fixed order

print("Starting tool call logging test...")
print(f"Graph: {len(np.unique(toolkit.clusters))} nodes, {toolkit.adjacency.shape[0]} timesteps")

# Start recording
toolkit.start_recording("test_tool_calls.rrd")
print("Recording started, initial graph logged")

# Get all tools
all_tools = toolkit.get_all_tools()
print(f"Available tools: {list(all_tools.keys())}")

# Call each tool exactly once in a fixed order
print("\n--- Calling tools in order ---")

# 1. node_distances_through_time
print("1. Calling node_distances_through_time(1, 3)...")
result = all_tools["node_distances_through_time"][0](node_id_1=1, node_id_2=3)
print(f"   Counter: {toolkit.call_counter}")

# 2. node_overlap_scores_through_time  
print("2. Calling node_overlap_scores_through_time(1, 3)...")
result = all_tools["node_overlap_scores_through_time"][0](node_id_1=1, node_id_2=3)
print(f"   Counter: {toolkit.call_counter}")

# 3. node_overlap_position_at_time
print("3. Calling node_overlap_position_at_time(1, 3, 5)...")
result = all_tools["node_overlap_position_at_time"][0](node_id_1=1, node_id_2=3, timestep=5)
print(f"   Counter: {toolkit.call_counter}")

# 4. inspect_highres_node_at_time
print("4. Calling inspect_highres_node_at_time(2, 10)...")
result = all_tools["inspect_highres_node_at_time"][0](node_id=2, timestep=10)
print(f"   Counter: {toolkit.call_counter}")

# 5. inspect_node_through_time
print("5. Calling inspect_node_through_time(2)...")
result = all_tools["inspect_node_through_time"][0](node_id=2)
print(f"   Counter: {toolkit.call_counter}")

# 6. inspect_scene_at_time
print("6. Calling inspect_scene_at_time(10)...")
result = all_tools["inspect_scene_at_time"][0](timestep=10)
print(f"   Counter: {toolkit.call_counter}")

# 7. voxelize_scene
print("7. Calling voxelize_scene(5)...")
result = all_tools["voxelize_scene"][0](timestep=5)
print(f"   Counter: {toolkit.call_counter}")

# 8. node_movement
print("8. Calling node_movement(2)...")
result = all_tools["node_movement"][0](node_id=2)
print(f"   Counter: {toolkit.call_counter}")

# Stop recording
toolkit.stop_recording()

Starting tool call logging test...
Graph: 11 nodes, 20 timesteps
Recording started, initial graph logged
Available tools: ['node_distances_through_time', 'node_overlap_scores_through_time', 'node_overlap_position_at_time', 'inspect_highres_node_at_time', 'inspect_node_through_time', 'inspect_scene_at_time', 'voxelize_scene']

--- Calling tools in order ---
1. Calling node_distances_through_time(1, 3)...
   Counter: 1
2. Calling node_overlap_scores_through_time(1, 3)...
   Counter: 2
3. Calling node_overlap_position_at_time(1, 3, 5)...
   Counter: 3
4. Calling inspect_highres_node_at_time(2, 10)...
   Counter: 4
5. Calling inspect_node_through_time(2)...
   Counter: 5
6. Calling inspect_scene_at_time(10)...
   Counter: 6
7. Calling voxelize_scene(5)...
   Counter: 7


In [23]:
# Cell 1: Check CUDA and SDPA backends
import torch
print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Compute capability: {torch.cuda.get_device_capability(0)}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")
print(f"SDPA flash: {torch.backends.cuda.flash_sdp_enabled()}")
print(f"SDPA mem_efficient: {torch.backends.cuda.mem_efficient_sdp_enabled()}")
print(f"SDPA math: {torch.backends.cuda.math_sdp_enabled()}")

CUDA device: NVIDIA A40
Compute capability: (8, 6)
cuDNN version: 90100
SDPA flash: True
SDPA mem_efficient: True
SDPA math: True


In [24]:
# Cell 2: Check model config
print(f"Attention impl: {model.config._attn_implementation}")
print(f"Model dtype: {model.dtype}")
print(f"Use cache: {model.generation_config.use_cache}")
print(f"Cache implementation: {model.generation_config.cache_implementation if hasattr(model.generation_config, 'cache_implementation') else 'default'}")

Attention impl: sdpa
Model dtype: torch.bfloat16
Use cache: True
Cache implementation: None


In [25]:
# Cell 3: Quick text-only benchmark (no vision overhead)
import time
inputs = processor.tokenizer("Hello, how are you?", return_tensors="pt").to(model.device)
torch.cuda.synchronize()
start = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
torch.cuda.synchronize()
elapsed = time.time() - start
n_tokens = out.shape[1] - inputs.input_ids.shape[1]
print(f"Text-only: {n_tokens / elapsed:.1f} tok/s ({n_tokens} tokens in {elapsed:.2f}s)")

Text-only: 25.4 tok/s (100 tokens in 3.94s)
